In [94]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.exceptions import ConvergenceWarning
import warnings

warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [95]:
# Constants
TRAIN_SIZE = 0.2
RANDOM_STATE = 1082

In [96]:
base_dir = "../../../Datasets/csv/UAVATTACK/"

In [97]:
# Load the dataset
file_names = [
    "spoofing-CTGAN-synthetic.csv",
    "spoofing-GaussianCopula-synthetic.csv",
    "spoofing-TVAE-synthetic.csv",
    "syntheticSpoofing_cgan_data.csv",
    "syntheticSpoofing_gan_data.csv"
]
dataframes = {}
for file_name in file_names:
    file_path = os.path.join(base_dir, file_name)
    try:
        dataframes[file_name] = pd.read_csv(file_path)
        print(f"Successfully loaded {file_name}")
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
    except Exception as e:
        print(f"An error occurred while loading {file_name}: {e}")


Successfully loaded spoofing-CTGAN-synthetic.csv
Successfully loaded spoofing-GaussianCopula-synthetic.csv
Successfully loaded spoofing-TVAE-synthetic.csv
Successfully loaded syntheticSpoofing_cgan_data.csv
Successfully loaded syntheticSpoofing_gan_data.csv


In [98]:
# Preprocess the datasets
processed_data = {}

print("\nStarting preprocessing for each dataset")

for name, df in dataframes.items():
    print(f"Processing: {name}")
    if 'label' not in df.columns:
        print(f"-> Skipping {name}: 'label' column not found.")
        continue

    X = df.drop('label', axis=1)
    y = df['label']
    if y.dtype == object or y.dtype.name == 'category':
        le = LabelEncoder()
        y = le.fit_transform(y)

    print(f"Features shape: {X.shape}, Target shape: {y.shape}")

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=TRAIN_SIZE, random_state=RANDOM_STATE,
                                                        stratify=y)

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Convert scaled arrays back to DataFrames
    X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
    X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns)

    # Store all the results in a dictionary
    processed_data[name] = {
        'X_train_scaled': X_train_scaled_df,
        'y_train': y_train,
        'X_test_scaled': X_test_scaled_df,
        'y_test': y_test,
        'scaler': scaler
    }


Starting preprocessing for each dataset
Processing: spoofing-CTGAN-synthetic.csv
Features shape: (10000, 82), Target shape: (10000,)
Processing: spoofing-GaussianCopula-synthetic.csv
Features shape: (10000, 82), Target shape: (10000,)
Processing: spoofing-TVAE-synthetic.csv
Features shape: (10000, 82), Target shape: (10000,)
Processing: syntheticSpoofing_cgan_data.csv
Features shape: (2897, 82), Target shape: (2897,)
Processing: syntheticSpoofing_gan_data.csv
Features shape: (10000, 82), Target shape: (10000,)


In [99]:
# Models to evaluate
models = {
    "Logistic Regression": LogisticRegression(random_state=RANDOM_STATE, max_iter=5000, solver='saga'),
    "MLP Classifier": MLPClassifier(hidden_layer_sizes=(100, 50, 25), activation='tanh', solver='adam',
                                    random_state=RANDOM_STATE, learning_rate='adaptive', early_stopping=True, ),
    "XGBoost": XGBClassifier(objective='binary:logistic', random_state=RANDOM_STATE, eval_metric='mae', )
}

In [100]:
for name, data in processed_data.items():
    print(f"Running models on: {name}")

    # Get the data splits from dictionary
    X_train = data['X_train_scaled']
    y_train = data['y_train']
    X_test = data['X_test_scaled']
    y_test = data['y_test']

    # loop through each model, train it, and evaluate its performance
    for model_name, model in models.items():
        print(f"\n--- {model_name} ---")

        # Train the model
        print("Training model")
        model.fit(X_train, y_train)

        # Make predictions on the test set
        print("Evaluating model")
        y_pred = model.predict(X_test)

        # Print the evaluation metrics
        accuracy = accuracy_score(y_test, y_pred)
        print(f"Accuracy: {accuracy:.4f}")

        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))

        print("Confusion Matrix:")
        print(confusion_matrix(y_test, y_pred))


Running models on: spoofing-CTGAN-synthetic.csv

--- Logistic Regression ---
Training model
Evaluating model
Accuracy: 0.9875

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      3335
           1       0.99      0.99      0.99      4665

    accuracy                           0.99      8000
   macro avg       0.99      0.99      0.99      8000
weighted avg       0.99      0.99      0.99      8000

Confusion Matrix:
[[3292   43]
 [  57 4608]]

--- MLP Classifier ---
Training model
Evaluating model
Accuracy: 0.9870

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      3335
           1       0.99      0.99      0.99      4665

    accuracy                           0.99      8000
   macro avg       0.99      0.99      0.99      8000
weighted avg       0.99      0.99      0.99      8000

Confusion Matrix:
[[3296   39]
 [  65 4600]]

--- XGBo